In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets scikit-learn
!pip install -q -U --force-reinstall pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.3 MB/s eta 0:00:00


In [5]:
import shutil
shutil.copy("/content/Stanza_AI_Intent_Classification_Final_700.csv", "/content/drive/MyDrive/Stanza_AI_Intent_Classification_Final_700.csv")

'/content/drive/MyDrive/Stanza_AI_Intent_Classification_Final_700.csv'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Loading Dataset

In [6]:
import pandas as pd
import json

df = pd.read_csv("/content/drive/MyDrive/Stanza_AI_Intent_Classification_Final_700.csv")

def build_prompt(query):
    return f"""You are an intent classifier for a hotel AI system called Stanza AI. Given a guest or staff query, identify the responsible agent and the specific intent.

Query: {query}
Answer (JSON only):"""

def build_target(agent_label, intent_name):
    return json.dumps({"agent_label": agent_label, "intent_name": intent_name})

df["prompt"] = df["query"].apply(build_prompt)
df["target"] = df.apply(lambda r: build_target(r["agent_label"], r["intent_name"]), axis=1)
df["full_text"] = df["prompt"] + " " + df["target"]

test_df = df[df["split"] == "test"].reset_index(drop=True)
print("Test size:", len(test_df))

Test size: 70


Loading Base Qwen Model (Quantized, No Training)

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Base model loaded.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Base model loaded.


Loading Trained Adapter (No Training Just Load)

In [8]:
from peft import PeftModel

model = PeftModel.from_pretrained(model, "/content/drive/MyDrive/stanza_ai_qwen_checkpoint")
model = model.eval()
print("Trained Qwen adapter loaded successfully!")

Trained Qwen adapter loaded successfully!


Evaluation on Test Set

In [9]:
import re, json as jsonlib
from tqdm import tqdm

def generate_answer(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

def extract_json(text):
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        try:
            return jsonlib.loads(match.group())
        except:
            return None
    return None

results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    raw_output = generate_answer(row["prompt"])
    parsed = extract_json(raw_output)
    results.append({
        "true_agent": row["agent_label"], "true_intent": row["intent_name"],
        "pred_agent": parsed.get("agent_label") if parsed else None,
        "pred_intent": parsed.get("intent_name") if parsed else None,
    })

qwen_results_df = pd.DataFrame(results)

100%|██████████| 70/70 [07:19<00:00,  6.28s/it]


Accuracy

In [10]:
qwen_results_df["fully_correct"] = (qwen_results_df["pred_agent"] == qwen_results_df["true_agent"]) & (qwen_results_df["pred_intent"] == qwen_results_df["true_intent"])
qwen_overall = qwen_results_df["fully_correct"].mean() * 100
qwen_agent_acc = (qwen_results_df["pred_agent"] == qwen_results_df["true_agent"]).mean() * 100
qwen_intent_acc = (qwen_results_df["pred_intent"] == qwen_results_df["true_intent"]).mean() * 100

print(f"QWEN — Overall: {qwen_overall:.2f}% | Agent: {qwen_agent_acc:.2f}% | Intent: {qwen_intent_acc:.2f}%")

QWEN — Overall: 82.86% | Agent: 94.29% | Intent: 82.86%
